Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


Install independencies

In [2]:
!pip install anthropic --break-system-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 39.5 MB/s eta 0:00:00


Test

In [3]:
import anthropic

client = userdata.get('GOOGLE_API_KEY')

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=10,
    messages=[{"role": "user", "content": "Say hello."}]
)
print(response.content[0].text)

Hello! 👋 How can I help


For one image

In [11]:
import anthropic
import base64
import re
import os

os.environ["PYTHONIOENCODING"] = "utf-8"

client = userdata.get('GOOGLE_API_KEY')

image_path = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_images/9.jpg"

with open(image_path, "rb") as f:
    image_data = base64.standard_b64encode(f.read()).decode("utf-8")

prompt_text = """You are an expert in classifying harmful memes in the Chinese language. Your objective is to assess whether a meme is harmful or not.

Input: [Meme]
Image: [See attached image]
Text embedded: [Read from the image above]

Follow the steps below:
Step 1: Analyze the input meme by assessing the image and text to determine its harmfulness.
Step 2: If the meme contains any negative or insulting reference to gay or lesbian people, output Homophobia.
Step 3: If the meme contains any negative or insulting reference to transgender people, output Transphobia.
Step 4: If neither of the above applies, output Non_LGBT.

Output:
Your output should strictly follow the format:
Class labels: Homophobia, Transphobia, or Non_LGBT
Thought: Give your reason here"""

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=300,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/jpeg",
                        "data": image_data
                    }
                },
                {
                    "type": "text",
                    "text": prompt_text
                }
            ]
        }
    ]
)

raw = response.content[0].text.strip()
print("RAW OUTPUT:", raw)

# 先找 Class labels 那行
m = re.search(r"Class labels?:\s*(Homophobia|Transphobia|Non_LGBT)", raw, re.IGNORECASE)
if m:
    label = m.group(1)
elif "transph" in raw.lower():
    label = "Transphobia"
elif "homoph" in raw.lower():
    label = "Homophobia"
else:
    label = None
print("PARSED LABEL:", label)

RAW OUTPUT: # Meme Classification Analysis

**Step 1: Analyze the meme content**

Image: A man giving a thumbs up gesture with a friendly expression.

Text (Traditional Chinese): "只要轉換性別就可以提高世界紀錄"
Translation: "As long as you change/switch gender, you can break world records"

**Step 2: Assess for homophobia**

The meme does not contain negative or insulting references to gay or lesbian people.

**Step 3: Assess for transphobia**

The meme contains a mocking insinuation that transgender individuals who transition do so to gain unfair competitive advantages in sports. This represents a negative stereotype and insulting reference to transgender people, implying their gender identity is illegitimate or a form of cheating.

**Step 4: Conclusion**

---

**Class labels:** Transphobia

**Thought:** The meme ridicules transgender people by suggesting that gender transition is merely a tactic to unfairly break athletic records. This perpetuates harmful stereotypes that delegitimize transgender 

For multi images

删除错误结果

In [ ]:
import json

with open("/content/drive/MyDrive/Gemini25Flash_HM_ZeroShot_pred.json", "r", encoding="utf-8") as f:
    predictions = json.load(f)

print(f"当前已保存: {len(predictions)} 条")

当前已保存: 17 条


In [13]:
import os
import json
import base64
import re
import time
import anthropic
from tqdm import tqdm

os.environ["PYTHONIOENCODING"] = "utf-8"

client = userdata.get('GOOGLE_API_KEY')

image_dir = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_images"
output_json = "/content/drive/MyDrive/ClaudeHaiku_HM_ZeroShot_pred.json"

prompt_text = """You are an expert in classifying harmful memes in the Chinese language. Your objective is to assess whether a meme is harmful or not.

Input: [Meme]
Image: [See attached image]
Text embedded: [Read from the image above]

Follow the steps below:
Step 1: Analyze the input meme by assessing the image and text to determine its harmfulness.
Step 2: If the meme contains any negative or insulting reference to gay or lesbian people, output Homophobia.
Step 3: If the meme contains any negative or insulting reference to transgender people, output Transphobia.
Step 4: If neither of the above applies, output Non_LGBT.

Output:
Your output should strictly follow the format:
Class labels: Homophobia, Transphobia, or Non_LGBT
Thought: Give your reason here"""

def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.standard_b64encode(f.read()).decode("utf-8")

def call_with_retry(image_data, media_type, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=300,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "image",
                                "source": {
                                    "type": "base64",
                                    "media_type": media_type,
                                    "data": image_data
                                }
                            },
                            {
                                "type": "text",
                                "text": prompt_text
                            }
                        ]
                    }
                ]
            )
            return response.content[0].text.strip()
        except Exception as e:
            if "429" in str(e) or "overloaded" in str(e).lower():
                wait = 30 * (attempt + 1)
                print(f"  限速，等待 {wait} 秒后重试...")
                time.sleep(wait)
            else:
                raise e
    raise Exception("超过最大重试次数")

image_files = sorted(
    [f for f in os.listdir(image_dir) if f.lower().endswith((".jpg", ".jpeg", ".png", ".gif"))],
    key=lambda x: int(re.search(r"(\d+)", x).group(1)) if re.search(r"(\d+)", x) else 0
)

# 断点续跑
if os.path.exists(output_json):
    with open(output_json, "r", encoding="utf-8") as f:
        predictions = json.load(f)
    predictions = [p for p in predictions if p["predicted_label"] != "ERROR"]
    done_images = {p["image_name"] for p in predictions}
    print(f"发现已有成功结果 {len(done_images)} 条，从断点继续...")
else:
    predictions = []
    done_images = set()
    print("没有已有结果，从头开始...")

remaining = [f for f in image_files if f not in done_images]
print(f"剩余待处理: {len(remaining)} 张")

for img_name in tqdm(remaining, desc="推理进度"):
    img_path = os.path.join(image_dir, img_name)

    try:
        ext = img_name.lower().split(".")[-1]
        media_type = "image/png" if ext == "png" else "image/gif" if ext == "gif" else "image/jpeg"

        image_data = encode_image(img_path)
        raw = call_with_retry(image_data, media_type)

        # 解析 label
        m = re.search(r"Class labels?:\s*(Homophobia|Transphobia|Non_LGBT)", raw, re.IGNORECASE)
        if m:
            label = m.group(1)
        elif any(w in raw.lower() for w in ["unable", "cannot", "can't", "sorry", "i can't help"]):
            label = "Non_LGBT"
        elif "transph" in raw.lower():
            label = "Transphobia"
        elif "homoph" in raw.lower():
            label = "Homophobia"
        else:
            label = "UNKNOWN"

        predictions.append({
            "image_name": img_name,
            "predicted_label": label,
            "raw_output": raw
        })

        print(f"✅ {img_name} -> {label}")

        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)

        time.sleep(1)

    except Exception as e:
        print(f"❌ {img_name} 出错: {e}")
        predictions.append({
            "image_name": img_name,
            "predicted_label": "ERROR",
            "raw_output": str(e)
        })
        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)
        time.sleep(3)

print(f"\n完成！共 {len(predictions)} 条结果已保存")
labels = [p["predicted_label"] for p in predictions]
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

发现已有成功结果 0 条，从断点继续...
剩余待处理: 232 张


推理进度:   0%|          | 0/232 [00:00<?, ?it/s]

✅ 1.jpg -> Transphobia


推理进度:   0%|          | 1/232 [00:04<19:08,  4.97s/it]

✅ 2.jpg -> Transphobia


...

✅ 238.jpg -> Non_LGBT


推理进度: 100%|█████████▉| 231/232 [16:09<00:05,  5.17s/it]

✅ 239.jpg -> Transphobia


推理进度: 100%|██████████| 232/232 [16:13<00:00,  4.20s/it]


完成！共 232 条结果已保存
  Homophobia: 41
  Non_LGBT: 50
  Transphobia: 118
  UNKNOWN: 23


Re-run unknown pics

In [16]:
import os
import json
import base64
import re
import time
import anthropic

os.environ["PYTHONIOENCODING"] = "utf-8"
client = userdata.get('GOOGLE_API_KEY')

output_json = "/content/drive/MyDrive/ClaudeHaiku_HM_ZeroShot_pred.json"
image_dir = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_images"

with open(output_json, "r", encoding="utf-8") as f:
    predictions = json.load(f)

unknowns = [p for p in predictions if p["predicted_label"] == "UNKNOWN"]
print(f"需要重跑: {len(unknowns)} 张")

for p in unknowns:
    img_name = p["image_name"]
    img_path = os.path.join(image_dir, img_name)
    print(f"重跑: {img_name}")

    for attempt in range(5):
        try:
            ext = img_name.lower().split(".")[-1]
            media_type = "image/png" if ext == "png" else "image/gif" if ext == "gif" else "image/jpeg"

            with open(img_path, "rb") as f:
                image_data = base64.standard_b64encode(f.read()).decode("utf-8")

            response = client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=500,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "image",
                                "source": {
                                    "type": "base64",
                                    "media_type": media_type,
                                    "data": image_data
                                }
                            },
                            {
                                "type": "text",
                                "text": prompt_text
                            }
                        ]
                    }
                ]
            )

            raw = response.content[0].text.strip()

            m = re.search(r"Class labels?:\s*(Homophobia|Transphobia|Non_LGBT)", raw, re.IGNORECASE)
            if m:
                label = m.group(1)
            elif any(w in raw.lower() for w in ["unable", "cannot", "can't", "sorry", "i can't help"]):
                label = "Non_LGBT"
            elif "transph" in raw.lower():
                label = "Transphobia"
            elif "homoph" in raw.lower():
                label = "Homophobia"
            else:
                label = "UNKNOWN"

            p["predicted_label"] = label
            p["raw_output"] = raw
            print(f"✅ {img_name} -> {label}")
            time.sleep(1)
            break

        except Exception as e:
            print(f"  第{attempt+1}次失败: {str(e)[:100]}")
            time.sleep(30)

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

labels = [p["predicted_label"] for p in predictions]
print(f"\n最终统计:")
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

需要重跑: 23 张
重跑: 4.jpg
✅ 4.jpg -> Transphobia
重跑: 12.jpg
✅ 12.jpg -> Non_LGBT
重跑: 47.jpg
✅ 47.jpg -> Transphobia
重跑: 61.jpeg
✅ 61.jpeg -> Homophobia
重跑: 70.jpg
✅ 70.jpg -> Transphobia
重跑: 78.jpg
✅ 78.jpg -> UNKNOWN
重跑: 130.jpg
✅ 130.jpg -> Transphobia
重跑: 139.jpg
✅ 139.jpg -> Transphobia
重跑: 140.jpg
✅ 140.jpg -> Non_LGBT
重跑: 142.jpeg
✅ 142.jpeg -> Homophobia
重跑: 147.jpg
✅ 147.jpg -> UNKNOWN
重跑: 160.jpg
✅ 160.jpg -> Non_LGBT
重跑: 170.jpg
✅ 170.jpg -> UNKNOWN
重跑: 172.jpg
✅ 172.jpg -> Homophobia
重跑: 175.jpg
✅ 175.jpg -> Transphobia
重跑: 178.jpg
✅ 178.jpg -> Transphobia
重跑: 190.jpg
✅ 190.jpg -> Transphobia
重跑: 198.jpg
✅ 198.jpg -> UNKNOWN
重跑: 199.gif
✅ 199.gif -> Transphobia
重跑: 202.jpg
✅ 202.jpg -> Homophobia
重跑: 204.jpg
✅ 204.jpg -> Transphobia
重跑: 213.jpg
✅ 213.jpg -> Transphobia
重跑: 236.jpg
✅ 236.jpg -> UNKNOWN

最终统计:
  Homophobia: 45
  Non_LGBT: 53
  Transphobia: 129
  UNKNOWN: 5


In [17]:
import json

with open("/content/drive/MyDrive/ClaudeHaiku_HM_ZeroShot_pred.json", "r", encoding="utf-8") as f:
    predictions = json.load(f)

unknowns = [p for p in predictions if p["predicted_label"] == "UNKNOWN"]
print(f"UNKNOWN 数量: {len(unknowns)}")
for u in unknowns[:3]:
    print(f"\n图片: {u['image_name']}")
    print(f"原始输出: {u['raw_output'][:300]}")
    print()

UNKNOWN 数量: 5

图片: 78.jpg
原始输出: # Analysis of Meme Content

**Step 1: Image and Text Analysis**

The image depicts a character in traditional Chinese styling with distinctive features (dramatic black hair, red and gold clothing, holding a staff).

The embedded Chinese text reads:
- Top left: "表明立场，只爱教因" (Clarify position, only lov


图片: 147.jpg
原始输出: # Analysis

**Step 1: Content Assessment**

This image shows a product label in Chinese with the following information:
- Product name: 玻璃白醋 (Glass White Vinegar)
- Packaging date: 2024.02.10
- Expiration date: 2025.02.09
- Net weight: 0.086 kg
- Price per kg: 140.00/kg
- Total price: 12.0...
- A ba


图片: 170.jpg
原始输出: # Analysis

**Step 1: Image and Text Assessment**

The image shows a storefront sign in Traditional Chinese characters on a teal/green background. The visible text reads: "部◇百合中醫堂" (with part of the first character cut off on the left).

Breaking down the characters:
- 部 (department/section)
- ◇ (de



In [18]:
import json

with open("/content/drive/MyDrive/ClaudeHaiku_HM_ZeroShot_pred.json", "r", encoding="utf-8") as f:
    predictions = json.load(f)

unknowns = [p for p in predictions if p["predicted_label"] == "UNKNOWN"]
for u in unknowns:
    print(f"图片: {u['image_name']}")
    print(f"输出长度: {len(u['raw_output'])}")
    print(f"最后50字符: {u['raw_output'][-50:]}")
    print()

图片: 78.jpg
输出长度: 1266
最后50字符: an mockery or stigmatization of LGBT+ individuals.

图片: 147.jpg
输出长度: 1003
最后50字符: any insulting language toward any protected group.

图片: 170.jpg
输出长度: 1231
最后50字符: gender people. It is a neutral commercial signage.

图片: 198.jpg
输出长度: 936
最后50字符:  harmless and non-targeted at any protected group.

图片: 236.jpg
输出长度: 853
最后50字符: up based on sexual orientation or gender identity.



In [19]:
import json

output_json = "/content/drive/MyDrive/ClaudeHaiku_HM_ZeroShot_pred.json"

with open(output_json, "r", encoding="utf-8") as f:
    predictions = json.load(f)

for p in predictions:
    if p["predicted_label"] == "UNKNOWN":
        p["predicted_label"] = "Non_LGBT"
        print(f"修正: {p['image_name']} -> Non_LGBT")

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

labels = [p["predicted_label"] for p in predictions]
print(f"\n最终统计:")
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

修正: 78.jpg -> Non_LGBT
修正: 147.jpg -> Non_LGBT
修正: 170.jpg -> Non_LGBT
修正: 198.jpg -> Non_LGBT
修正: 236.jpg -> Non_LGBT

最终统计:
  Homophobia: 45
  Non_LGBT: 58
  Transphobia: 129


Calculate Zero-shot result

In [20]:
import json
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

json_path = "/content/drive/MyDrive/ClaudeHaiku_HM_ZeroShot_pred.json"

with open(json_path, "r", encoding="utf-8") as f:
    predictions = json.load(f)

pred_df = pd.DataFrame(predictions)
pred_df["image_id"] = (
    pred_df["image_name"]
    .str.replace(".jpg", "", regex=False)
    .str.replace(".jpeg", "", regex=False)
    .str.replace(".png", "", regex=False)
    .str.replace(".gif", "", regex=False)
    .str.strip()
)

excel_path = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_labels.xlsx"
true_df = pd.read_excel(excel_path)
true_df.columns = true_df.columns.str.strip().str.lower()
true_df["image_id"] = true_df["id"].astype(str).str.strip()

merged_df = pd.merge(true_df, pred_df, on="image_id", how="inner")

print("Total True Labels:", len(true_df))
print("Total Predictions:", len(pred_df))
print("After Merge:", len(merged_df))

def normalize_label(x):
    x = str(x).strip().lower()
    if x in ["homophobia", "homophobic"]:
        return "homophobic"
    if x in ["transphobia", "transphobic"]:
        return "transphobic"
    if x in ["non_lgbt", "non-above", "non_anti_lgbt", "non anti lgbt"]:
        return "non anti lgbt"
    return x

merged_df["true_label"] = merged_df["label"].apply(normalize_label)
merged_df["predicted_label"] = merged_df["predicted_label"].apply(normalize_label)

print("\nUnique TRUE labels:", sorted(merged_df["true_label"].unique()))
print("Unique PRED labels:", sorted(merged_df["predicted_label"].unique()))

y_true = merged_df["true_label"]
y_pred = merged_df["predicted_label"]

ACC = accuracy_score(y_true, y_pred)
MP  = precision_score(y_true, y_pred, average="macro", zero_division=0)
MR  = recall_score(y_true, y_pred, average="macro", zero_division=0)
MF1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
WP  = precision_score(y_true, y_pred, average="weighted", zero_division=0)
WR  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
WF1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("\nEvaluation Metrics")
print("-------------------------------------------------")
print(f"ACC  : {ACC:.4f}")
print(f"MP   : {MP:.4f}")
print(f"MR   : {MR:.4f}")
print(f"MF1  : {MF1:.4f}")
print(f"WP   : {WP:.4f}")
print(f"WR   : {WR:.4f}")
print(f"WF1  : {WF1:.4f}")
print("-------------------------------------------------")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

Total True Labels: 239
Total Predictions: 232
After Merge: 232

Unique TRUE labels: ['homophobic', 'non anti lgbt', 'transphobic']
Unique PRED labels: ['homophobic', 'non anti lgbt', 'transphobic']

Evaluation Metrics
-------------------------------------------------
ACC  : 0.2974
MP   : 0.4315
MR   : 0.4536
MF1  : 0.2818
WP   : 0.7433
WR   : 0.2974
WF1  : 0.3584
-------------------------------------------------

Confusion Matrix:
[[42 41 86]
 [ 1 16 32]
 [ 2  1 11]]

Classification Report:
               precision    recall  f1-score   support

   homophobic       0.93      0.25      0.39       169
non anti lgbt       0.28      0.33      0.30        49
  transphobic       0.09      0.79      0.15        14

     accuracy                           0.30       232
    macro avg       0.43      0.45      0.28       232
 weighted avg       0.74      0.30      0.36       232



Few-shot with multiple pics

In [4]:
import os
import json
import base64
import re
import time
import numpy as np
import torch
import anthropic
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel

os.environ["PYTHONIOENCODING"] = "utf-8"
client = userdata.get('GOOGLE_API_KEY')

# ===============================
# 路径配置
# ===============================
test_image_dir = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_images"
output_json = "/content/drive/MyDrive/ClaudeHaiku_HM_FewShot_RAG_pred.json"

# ===============================
# 加载 CLIP + 训练集 embeddings
# ===============================
print("Loading CLIP...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

train_embeddings = np.load("/content/drive/MyDrive/train_embeddings.npy")
with open("/content/drive/MyDrive/train_meta.json", "r") as f:
    train_meta = json.load(f)

train_labels = train_meta["labels"]
train_filenames = train_meta["filenames"]
print(f"训练集 embeddings 加载完成，共 {len(train_embeddings)} 条")

# ===============================
# RAG 检索函数
# ===============================
def get_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = clip_model.vision_model(**inputs)
        emb = outputs.pooler_output
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.squeeze().numpy()

def retrieve_examples(test_emb):
    similarities = train_embeddings @ test_emb
    examples = {}
    for label in ["Homophobic", "Transphobic", "Non_Anti_LGBT"]:
        label_indices = [i for i, l in enumerate(train_labels) if l == label]
        label_sims = [(i, similarities[i]) for i in label_indices]
        label_sims.sort(key=lambda x: x[1], reverse=True)
        examples[label] = label_sims[0][0]
    return examples

def build_prompt(example_indices):
    label_map = {
        "Homophobic": "Homophobia",
        "Transphobic": "Transphobia",
        "Non_Anti_LGBT": "Non_LGBT"
    }
    examples_text = ""
    for i, (label, idx) in enumerate(example_indices.items()):
        examples_text += f"Example {i+1}: Class label: {label_map[label]}\n"

    return f"""You are an expert in classifying harmful memes in the Chinese language. Your objective is to assess whether a meme is harmful or not.

Below are 3 reference examples with their correct labels retrieved from similar memes:

{examples_text}
Now classify the following meme:

Input: [Meme]
Image: [See attached image]
Text embedded: [Read from the image above]

Follow the steps below:
Step 1: Analyze the input meme by assessing the image and text, using the provided examples as reference to determine its harmfulness.
Step 2: If the meme contains any negative or insulting reference to gay or lesbian people, output Homophobia.
Step 3: If the meme contains any negative or insulting reference to transgender people, output Transphobia.
Step 4: If neither of the above applies, output Non_LGBT.

Output:
Your output should strictly follow the format:
Class labels: Homophobia, Transphobia, or Non_LGBT
Thought: Give your reason here"""

# ===============================
# API 调用（带重试）
# ===============================
def call_with_retry(image_data, media_type, prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=500,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "image",
                                "source": {
                                    "type": "base64",
                                    "media_type": media_type,
                                    "data": image_data
                                }
                            },
                            {
                                "type": "text",
                                "text": prompt
                            }
                        ]
                    }
                ]
            )
            return response.content[0].text.strip()
        except Exception as e:
            if "429" in str(e) or "overloaded" in str(e).lower():
                wait = 30 * (attempt + 1)
                print(f"  限速，等待 {wait} 秒后重试...")
                time.sleep(wait)
            else:
                raise e
    raise Exception("超过最大重试次数")

# ===============================
# 获取测试图片列表
# ===============================
image_files = sorted(
    [f for f in os.listdir(test_image_dir) if f.lower().endswith((".jpg", ".jpeg", ".png", ".gif"))],
    key=lambda x: int(re.search(r"(\d+)", x).group(1)) if re.search(r"(\d+)", x) else 0
)

# 断点续跑
if os.path.exists(output_json):
    with open(output_json, "r", encoding="utf-8") as f:
        predictions = json.load(f)
    predictions = [p for p in predictions if p["predicted_label"] != "ERROR"]
    done_images = {p["image_name"] for p in predictions}
    print(f"发现已有成功结果 {len(done_images)} 条，从断点继续...")
else:
    predictions = []
    done_images = set()
    print("没有已有结果，从头开始...")

remaining = [f for f in image_files if f not in done_images]
print(f"剩余待处理: {len(remaining)} 张")

# ===============================
# 批量推理
# ===============================
for img_name in tqdm(remaining, desc="推理进度"):
    img_path = os.path.join(test_image_dir, img_name)

    try:
        # RAG 检索
        test_emb = get_embedding(img_path)
        example_indices = retrieve_examples(test_emb)
        prompt_text = build_prompt(example_indices)

        ext = img_name.lower().split(".")[-1]
        media_type = "image/png" if ext == "png" else "image/gif" if ext == "gif" else "image/jpeg"

        with open(img_path, "rb") as f:
            image_data = base64.standard_b64encode(f.read()).decode("utf-8")

        raw = call_with_retry(image_data, media_type, prompt_text)

        m = re.search(r"Class labels?:\s*(Homophobia|Transphobia|Non_LGBT)", raw, re.IGNORECASE)
        if m:
            label = m.group(1)
        elif any(w in raw.lower() for w in ["unable", "cannot", "can't", "sorry", "i can't help"]):
            label = "Non_LGBT"
        elif "transph" in raw.lower():
            label = "Transphobia"
        elif "homoph" in raw.lower():
            label = "Homophobia"
        else:
            label = "UNKNOWN"

        predictions.append({
            "image_name": img_name,
            "predicted_label": label,
            "raw_output": raw
        })

        print(f"✅ {img_name} -> {label}")

        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)

        time.sleep(1)

    except Exception as e:
        print(f"❌ {img_name} 出错: {e}")
        predictions.append({
            "image_name": img_name,
            "predicted_label": "ERROR",
            "raw_output": str(e)
        })
        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)
        time.sleep(3)

print(f"\n完成！共 {len(predictions)} 条结果已保存")
labels = [p["predicted_label"] for p in predictions]
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

Loading CLIP...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

✅ 238.jpg -> Non_LGBT


推理进度: 100%|█████████▉| 231/232 [19:15<00:05,  5.43s/it]

✅ 239.jpg -> Transphobia


推理进度: 100%|██████████| 232/232 [19:20<00:00,  5.00s/it]


完成！共 232 条结果已保存
  Homophobia: 57
  Non_LGBT: 107
  Transphobia: 57
  UNKNOWN: 11


In [5]:
import json

with open("/content/drive/MyDrive/ClaudeHaiku_HM_FewShot_RAG_pred.json", "r", encoding="utf-8") as f:
    predictions = json.load(f)

unknowns = [p for p in predictions if p["predicted_label"] == "UNKNOWN"]
print(f"UNKNOWN 数量: {len(unknowns)}")
for u in unknowns:
    print(f"\n图片: {u['image_name']}")
    print(f"输出长度: {len(u['raw_output'])}")
    print(f"最后100字符: {u['raw_output'][-100:]}")
    print()

UNKNOWN 数量: 11

图片: 17.jpg
输出长度: 1256
最后100字符: y toward LGBT+ individuals. It is purely descriptive content about physical appearance and grooming.


图片: 18.jpeg
输出长度: 1127
最后100字符: s. The critique is generalized social commentary rather than targeted harassment of any LGBT+ group.


图片: 43.jpg
输出长度: 1231
最后100字符: guishing true friends from superficial contacts, unrelated to sexual orientation or gender identity.


图片: 72.jpg
输出长度: 913
最后100字符: egative, insulting, or derogatory references to any LGBT individuals or groups present in the image.


图片: 91.jpg
输出长度: 457
最后100字符:  gender identity, or any LGBT-related topics. This is a general humor meme unrelated to LGBT issues.


图片: 99.jpeg
输出长度: 809
最后100字符: o sexual orientation or gender identity.

Therefore, this meme should be classified as **Non_LGBT**.


图片: 140.jpg
输出长度: 1072
最后100字符: dentity. The content is benign and focuses on expressing affection and well-wishes to the recipient.


图片: 154.jpg
输出长度: 581
最后100字符: image centered 

In [6]:
import json

output_json = "/content/drive/MyDrive/ClaudeHaiku_HM_FewShot_RAG_pred.json"

with open(output_json, "r", encoding="utf-8") as f:
    predictions = json.load(f)

for p in predictions:
    if p["predicted_label"] == "UNKNOWN":
        p["predicted_label"] = "Non_LGBT"
        print(f"修正: {p['image_name']} -> Non_LGBT")

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

labels = [p["predicted_label"] for p in predictions]
print(f"\n最终统计:")
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

修正: 17.jpg -> Non_LGBT
修正: 18.jpeg -> Non_LGBT
修正: 43.jpg -> Non_LGBT
修正: 72.jpg -> Non_LGBT
修正: 91.jpg -> Non_LGBT
修正: 99.jpeg -> Non_LGBT
修正: 140.jpg -> Non_LGBT
修正: 154.jpg -> Non_LGBT
修正: 190.jpg -> Non_LGBT
修正: 204.jpg -> Non_LGBT
修正: 210.jpg -> Non_LGBT

最终统计:
  Homophobia: 57
  Non_LGBT: 118
  Transphobia: 57


 Calculation of Few shot

In [8]:
import json
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

json_path = "/content/drive/MyDrive/ClaudeHaiku_HM_FewShot_RAG_pred.json"

with open(json_path, "r", encoding="utf-8") as f:
    predictions = json.load(f)

pred_df = pd.DataFrame(predictions)
pred_df["image_id"] = (
    pred_df["image_name"]
    .str.replace(".jpg", "", regex=False)
    .str.replace(".jpeg", "", regex=False)
    .str.replace(".png", "", regex=False)
    .str.replace(".gif", "", regex=False)
    .str.strip()
)

excel_path = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_labels.xlsx"
true_df = pd.read_excel(excel_path)
true_df.columns = true_df.columns.str.strip().str.lower()
true_df["image_id"] = true_df["id"].astype(str).str.strip()

merged_df = pd.merge(true_df, pred_df, on="image_id", how="inner")

print("Total True Labels:", len(true_df))
print("Total Predictions:", len(pred_df))
print("After Merge:", len(merged_df))

def normalize_label(x):
    x = str(x).strip().lower()
    if x in ["homophobia", "homophobic"]:
        return "homophobic"
    if x in ["transphobia", "transphobic"]:
        return "transphobic"
    if x in ["non_lgbt", "non-above", "non_anti_lgbt", "non anti lgbt"]:
        return "non anti lgbt"
    return x

merged_df["true_label"] = merged_df["label"].apply(normalize_label)
merged_df["predicted_label"] = merged_df["predicted_label"].apply(normalize_label)

print("\nUnique TRUE labels:", sorted(merged_df["true_label"].unique()))
print("Unique PRED labels:", sorted(merged_df["predicted_label"].unique()))

y_true = merged_df["true_label"]
y_pred = merged_df["predicted_label"]

ACC = accuracy_score(y_true, y_pred)
MP  = precision_score(y_true, y_pred, average="macro", zero_division=0)
MR  = recall_score(y_true, y_pred, average="macro", zero_division=0)
MF1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
WP  = precision_score(y_true, y_pred, average="weighted", zero_division=0)
WR  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
WF1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("\nEvaluation Metrics")
print("-------------------------------------------------")
print(f"ACC  : {ACC:.4f}")
print(f"MP   : {MP:.4f}")
print(f"MR   : {MR:.4f}")
print(f"MF1  : {MF1:.4f}")
print(f"WP   : {WP:.4f}")
print(f"WR   : {WR:.4f}")
print(f"WF1  : {WF1:.4f}")
print("-------------------------------------------------")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

Total True Labels: 239
Total Predictions: 232
After Merge: 232

Unique TRUE labels: ['homophobic', 'non anti lgbt', 'transphobic']
Unique PRED labels: ['homophobic', 'non anti lgbt', 'transphobic']

Evaluation Metrics
-------------------------------------------------
ACC  : 0.4224
MP   : 0.4643
MR   : 0.5200
MF1  : 0.3717
WP   : 0.7747
WR   : 0.4224
WF1  : 0.4575
-------------------------------------------------

Confusion Matrix:
[[55 77 37]
 [ 0 36 13]
 [ 2  5  7]]

Classification Report:
               precision    recall  f1-score   support

   homophobic       0.96      0.33      0.49       169
non anti lgbt       0.31      0.73      0.43        49
  transphobic       0.12      0.50      0.20        14

     accuracy                           0.42       232
    macro avg       0.46      0.52      0.37       232
 weighted avg       0.77      0.42      0.46       232

